<a href="https://colab.research.google.com/github/fulvio-f/aerospace-course/blob/main/tutorial/Workshop_S%C3%A1bado_Aeroespacial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aerospace Telemetry: Live Mars Weather Analysis
This notebook demonstrates a complete aerospace data pipeline: from fetching real-time environmental data via **NASA's Mars Weather API** to mathematical thermal modeling.

Developed as educational material for the **OBSAT (Brazilian Satellite Olympiad)**, it bridges the gap between high-level data ingestion and specialized engineering analysis.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt

print("Environment initialized. Ready to interface with NASA JPL APIs.")

## 1. Live Data Ingestion
We will interface with the Mars Science Laboratory (MSL) API to retrieve the latest telemetry from the Curiosity Rover.

In [ ]:
def fetch_live_mars_telemetry():
    """Fetches the latest Martian weather data directly from NASA."""
    API_URL = "https://mars.nasa.gov/rss/api/?feed=weather&category=msl&feedtype=json"

    try:
        response = requests.get(API_URL, timeout=15)
        response.raise_for_status()
        data = response.json()

        # Extract the most recent Sol (Martian Day)
        latest_sol = data['soles'][0]

        metrics = {
            "sol": latest_sol['sol'],
            "earth_date": latest_sol['terrestrial_date'],
            "max_temp": float(latest_sol['max_temp']),
            "min_temp": float(latest_sol['min_temp'])
        }

        print(f"Successfully connected to NASA. Latest Sol: {metrics['sol']} ({metrics['earth_date']})")
        return metrics

    except Exception as e:
        print(f"API Connection Error: {e}")
        # High-standard fallback for offline development
        return {"sol": "Unknown", "earth_date": "N/A", "max_temp": -15.0, "min_temp": -70.0}

# Execute ingestion
telemetry = fetch_live_mars_telemetry()
print(f"Temperature Bounds: {telemetry['min_temp']}°C to {telemetry['max_temp']}°C")

## 2. Diurnal Thermal Modeling
Martian temperatures vary significantly between day and night. We use a sinusoidal model to estimate the thermal state based on normalized Sol time (0.0 to 1.0).

In [ ]:
def estimate_temperature(max_t, min_t, current_time):
    """
    Applies a sinusoidal approximation to estimate temperature
    at a specific point in the Martian diurnal cycle.
    """
    delta = max_t - min_t
    # Normalizing sine wave to peak at mid-day (0.5)
    thermal_factor = (np.sin(current_time * 2 * np.pi - np.pi / 2) + 1) / 2
    return min_t + (delta * thermal_factor)

# Simulation parameters
current_sol_time = 0.45  # Mid-morning simulation
current_temp_estimate = estimate_temperature(
    telemetry['max_temp'],
    telemetry['min_temp'],
    current_sol_time
)

print(f"Estimated Temperature at Sol Time {current_sol_time}: {current_temp_estimate:.2f}°C")

## 3. Mission Visualization
Plotting the full daily temperature profile based on the live data retrieved.

In [ ]:
# Generate timeline (0 to 1 Sol)
timeline = np.linspace(0, 1, 500)
temp_profile = [estimate_temperature(telemetry['max_temp'], telemetry['min_temp'], t) for t in timeline]

plt.figure(figsize=(12, 6))
plt.plot(timeline, temp_profile, color='#e27b58', linewidth=2.5, label='Estimated Thermal Cycle')
plt.scatter(current_sol_time, current_temp_estimate, color='blue', zorder=5, label='Current State')

plt.title(f"Martian Diurnal Temperature - Sol {telemetry['sol']}", fontsize=14)
plt.xlabel("Normalized Sol Time (Dawn to Dusk)", fontsize=12)
plt.ylabel("Temperature (°C)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()